In [1]:
# import necessary libraries

import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# load prepared dataset

data = np.load("citeseer_prepared.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
train_indices = data["train_indices"]
test_indices = data["test_indices"]
node_ids = data["node_ids"]

In [3]:
# load the neighbors
with open("citeseer_neighbors.pkl", "rb") as f:
    neighbors = pickle.load(f)

In [4]:
# recreate the mapping
node_to_index = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

# check mapping values
print("Features:", X.shape)
print("Labels:", y.shape)
print("Train nodes:", len(train_indices))
print("Test nodes:", len(test_indices))
print("Number of neighbor sets:", len(neighbors))

Features: (3312, 3703)
Labels: (3312,)
Train nodes: 2649
Test nodes: 663
Number of neighbor sets: 3312


In [ ]:
# create node data

def get_node_data(node_id):
    center_index = node_to_index[node_id]

    center_features = X[center_index]

    neighbor_ids = list(neighbors[node_id])

    neighbor_features = np.array([
        X[node_to_index[neighbor_id]]
        for neighbor_id in neighbor_ids
    ])

    label = y[center_index]

    return center_features, neighbor_features, label

In [6]:
# test with the first node
center, neighbor_features, label = get_node_data(node_ids[0])

print("Center shape:", center.shape)
print("Neighbors shape:", neighbor_features.shape)
print("Label:", label)

Center shape: (3703,)
Neighbors shape: (12, 3703)
Label: 1


In [7]:
# create the dataset
class CiteseerDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        node_index = self.indices[idx]
        node_id = node_ids[node_index]

        center_features, neighbor_features, label = get_node_data(node_id)

        return center_features, neighbor_features, label

In [8]:
# create train and test datasets
train_dataset = CiteseerDataset(train_indices)
test_dataset = CiteseerDataset(test_indices)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 2649
Test samples: 663


In [9]:
# create custom collate function for DataLoader
def citeseer_collate(batch):
    centers = []
    neighbor_features = []
    labels = []

    for center, neighbor_set, label in batch:
        centers.append(
            torch.tensor(center, dtype=torch.float32)
        )

        neighbor_features.append(
            torch.tensor(neighbor_set, dtype=torch.float32)
        )

        labels.append(label)

    centers = torch.stack(centers)
    labels = torch.tensor(labels, dtype=torch.long)

    return centers, neighbor_features, labels

In [10]:
# create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=citeseer_collate
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=citeseer_collate
)

In [11]:
# test a batch from the train loader
centers, neighbor_features, labels = next(iter(train_loader))

print("Centers:", centers.shape)
print("Number of neighbor sets:", len(neighbor_features))

for i, neighbor_set in enumerate(neighbor_features[:4]):
    print(
        f"Neighbor set {i + 1}:",
        neighbor_set.shape
    )

print("Labels:", labels.shape)

Centers: torch.Size([16, 3703])
Number of neighbor sets: 16
Neighbor set 1: torch.Size([1, 3703])
Neighbor set 2: torch.Size([1, 3703])
Neighbor set 3: torch.Size([2, 3703])
Neighbor set 4: torch.Size([2, 3703])
Labels: torch.Size([16])


In [ ]:
# create the class
class CiteseerJanossy(nn.Module):
    def __init__(
        self,
        input_dim=3703,
        hidden_dim=64,
        num_permutations=10,
        num_classes=6
    ):
        super().__init__()

        self.num_permutations = num_permutations

        # Project each neighbor's 3703 features into 64 dimensions
        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim
        )

        # Order-sensitive sequence model
        self.order_sensitive = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Encode the center node
        self.center_embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, center, neighbor_features):

        # Project neighbors: [N, 3703] -> [N, 64]
        neighbor_features = self.input_projection(
            neighbor_features
        )

        permutation_outputs = []

        # Sample several permutations
        for _ in range(self.num_permutations):

            permutation = torch.randperm(
                neighbor_features.size(0)
            )

            shuffled_features = neighbor_features[
                permutation
            ]

            # Add batch dimension:
            # [N, 64] -> [1, N, 64]
            shuffled_features = shuffled_features.unsqueeze(0)

            # Process the ordered sequence
            _, hidden = self.order_sensitive(
                shuffled_features
            )

            # hidden: [1, 1, 64]
            # Select the final hidden representation
            pooled = hidden[0, 0]

            permutation_outputs.append(pooled)

        # Average over sampled permutations
        janossy_representation = torch.stack(
            permutation_outputs
        ).mean(dim=0)

        # Encode center node
        center_features = self.center_embedding(
            center
        )

        # Combine center + neighborhood representation
        combined = torch.cat(
            [
                center_features,
                janossy_representation
            ],
            dim=0
        )

        # Classification
        output = self.classifier(combined)

        return output

In [36]:
# create the model
model = CiteseerJanossy(
    num_permutations=5
)

print(model)

CiteseerJanossy(
  (input_projection): Linear(in_features=3703, out_features=64, bias=True)
  (order_sensitive): GRU(64, 64, batch_first=True)
  (center_embedding): Sequential(
    (0): Linear(in_features=3703, out_features=64, bias=True)
    (1): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=6, bias=True)
  )
)


In [37]:
# check trainable parameters
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

Trainable parameters: 507718


In [38]:
# verify dataset before forward pass
with open("citeseer_neighbors.pkl", "rb") as f:
    neighbors = pickle.load(f)

print("Neighbor dictionary type:", type(neighbors))
print("Number of nodes:", len(neighbors))

center, neighbor_features, label = train_dataset[0]

print("Center shape:", center.shape)
print("Neighbor features shape:", neighbor_features.shape)
print("Label:", label)

Neighbor dictionary type: <class 'dict'>
Number of nodes: 3312
Center shape: (3703,)
Neighbor features shape: (1, 3703)
Label: 0


In [39]:
# verify forward pass

center_tensor = torch.tensor(
    center,
    dtype=torch.float32
)

neighbor_tensor = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

output = model(
    center_tensor,
    neighbor_tensor
)

print("Center tensor:", center_tensor.shape)
print("Neighbor tensor:", neighbor_tensor.shape)
print("Output:", output.shape)
print("True label:", label)

Center tensor: torch.Size([3703])
Neighbor tensor: torch.Size([1, 3703])
Output: torch.Size([6])
True label: 0


In [40]:
# create loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [42]:
# train for 20 epochs
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for centers, neighbor_sets, labels in train_loader:

        optimizer.zero_grad()

        batch_loss = 0.0

        for center, neighbor_features, label in zip(
            centers,
            neighbor_sets,
            labels
        ):

            output = model(
                center,
                neighbor_features
            )

            loss = criterion(
                output.unsqueeze(0),
                label.unsqueeze(0)
            )

            batch_loss += loss

        batch_loss = batch_loss / len(neighbor_sets)

        batch_loss.backward()

        optimizer.step()

        total_loss += batch_loss.item()

    average_loss = (
        total_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- Loss: {average_loss:.4f}"
    )

Epoch 01/20 - Loss: 0.4743
Epoch 02/20 - Loss: 0.1614
Epoch 03/20 - Loss: 0.0510
Epoch 04/20 - Loss: 0.0314
Epoch 05/20 - Loss: 0.0158
Epoch 06/20 - Loss: 0.0138
Epoch 07/20 - Loss: 0.0093
Epoch 08/20 - Loss: 0.0057
Epoch 09/20 - Loss: 0.0118
Epoch 10/20 - Loss: 0.0055
Epoch 11/20 - Loss: 0.0080
Epoch 12/20 - Loss: 0.0043
Epoch 13/20 - Loss: 0.0017
Epoch 14/20 - Loss: 0.0016
Epoch 15/20 - Loss: 0.0012
Epoch 16/20 - Loss: 0.0012
Epoch 17/20 - Loss: 0.0006
Epoch 18/20 - Loss: 0.0005
Epoch 19/20 - Loss: 0.0003
Epoch 20/20 - Loss: 0.0002


In [43]:
# evaluate the test set

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for centers, neighbor_sets, labels in test_loader:

        for center, neighbor_features, label in zip(
            centers,
            neighbor_sets,
            labels
        ):

            output = model(
                center,
                neighbor_features
            )

            prediction = torch.argmax(output).item()

            all_predictions.append(prediction)
            all_labels.append(label.item())

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

Test accuracy: 0.7632
Test accuracy: 76.32%


In [44]:
# run confusion matrix and classification report

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion matrix:")
print(cm)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "AI",
            "Agents",
            "DB",
            "HCI",
            "IR",
            "ML"
        ]
    )
)

Confusion matrix:
[[ 23   6   7   3   0  11]
 [  3 107   3   3   1   2]
 [  2   2 109   2  15  10]
 [  1  13   0  78   9   1]
 [  0   3  11   4 103  13]
 [  4   7   3   2  16  86]]
              precision    recall  f1-score   support

          AI       0.70      0.46      0.55        50
      Agents       0.78      0.90      0.83       119
          DB       0.82      0.78      0.80       140
         HCI       0.85      0.76      0.80       102
          IR       0.72      0.77      0.74       134
          ML       0.70      0.73      0.71       118

    accuracy                           0.76       663
   macro avg       0.76      0.73      0.74       663
weighted avg       0.76      0.76      0.76       663



In [45]:
# test permutation behavior
model.eval()

center, neighbor_features, label = test_dataset[0]

center_tensor = torch.tensor(
    center,
    dtype=torch.float32
)

neighbor_tensor = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

with torch.no_grad():

    original_output = model(
        center_tensor,
        neighbor_tensor
    )

    original_probability = torch.softmax(
        original_output,
        dim=0
    )

print(
    "Original probabilities:",
    original_probability
)

Original probabilities: tensor([1.3090e-04, 9.0415e-07, 2.3794e-06, 7.9717e-04, 9.9890e-01, 1.6401e-04])


In [46]:
# test shuffled permutation behavior
permutation = torch.randperm(
    neighbor_tensor.size(0)
)

shuffled_tensor = neighbor_tensor[
    permutation
]

with torch.no_grad():

    shuffled_output = model(
        center_tensor,
        shuffled_tensor
    )

    shuffled_probability = torch.softmax(
        shuffled_output,
        dim=0
    )

difference = torch.abs(
    original_probability - shuffled_probability
)

print(
    "Shuffled probabilities:",
    shuffled_probability
)

print(
    "Maximum difference:",
    difference.max().item()
)

Shuffled probabilities: tensor([1.3090e-04, 9.0415e-07, 2.3794e-06, 7.9717e-04, 9.9890e-01, 1.6401e-04])
Maximum difference: 0.0


In [47]:
# choose node with several neighbors
for i in range(len(test_dataset)):

    center, neighbor_features, label = test_dataset[i]

    if len(neighbor_features) >= 5:
        print("Test dataset index:", i)
        print("Number of neighbors:", len(neighbor_features))
        print("True label:", label)
        break

Test dataset index: 9
Number of neighbors: 5
True label: 0


In [48]:
# permutation test
center, neighbor_features, label = test_dataset[i]

center_tensor = torch.tensor(
    center,
    dtype=torch.float32
)

neighbor_tensor = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

model.eval()

with torch.no_grad():

    original_output = model(
        center_tensor,
        neighbor_tensor
    )

    original_probability = torch.softmax(
        original_output,
        dim=0
    )

print("Number of neighbors:", neighbor_tensor.shape[0])
print("Original probabilities:", original_probability)

Number of neighbors: 5
Original probabilities: tensor([9.9709e-01, 1.3382e-06, 1.3656e-06, 2.8244e-06, 8.9890e-08, 2.9023e-03])


In [49]:
# test on shuffled probabilities
permutation = torch.randperm(
    neighbor_tensor.size(0)
)

shuffled_tensor = neighbor_tensor[
    permutation
]

with torch.no_grad():

    shuffled_output = model(
        center_tensor,
        shuffled_tensor
    )

    shuffled_probability = torch.softmax(
        shuffled_output,
        dim=0
    )

difference = torch.abs(
    original_probability - shuffled_probability
)

print("Shuffled probabilities:", shuffled_probability)
print("Maximum difference:", difference.max().item())

Shuffled probabilities: tensor([9.9877e-01, 1.9430e-06, 1.7894e-06, 3.7044e-06, 8.1610e-08, 1.2259e-03])
Maximum difference: 0.0016764396568760276


In [50]:
# test for 10 permutations

# Select a test node with at least 5 neighbors
for i in range(len(test_dataset)):

    center, neighbor_features, label = test_dataset[i]

    if len(neighbor_features) >= 5:
        break

center_tensor = torch.tensor(
    center,
    dtype=torch.float32
)

neighbor_tensor = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

model.eval()

# Project neighbors once
with torch.no_grad():

    projected_neighbors = model.input_projection(
        neighbor_tensor
    )

    # Create 5 fixed permutations
    torch.manual_seed(42)

    permutations = [
        torch.randperm(projected_neighbors.size(0))
        for _ in range(5)
    ]

    def janossy_representation(neighbor_features):

        outputs = []

        for permutation in permutations:

            shuffled_features = neighbor_features[
                permutation
            ]

            shuffled_features = shuffled_features.unsqueeze(0)

            _, hidden = model.order_sensitive(
                shuffled_features
            )

            pooled = hidden[0, 0]

            outputs.append(pooled)

        return torch.stack(outputs).mean(dim=0)

    # Original ordering
    original_representation = janossy_representation(
        projected_neighbors
    )

    # Create a new ordering of the SAME neighbors
    shuffle = torch.randperm(
        projected_neighbors.size(0)
    )

    shuffled_projected = projected_neighbors[
        shuffle
    ]

    shuffled_representation = janossy_representation(
        shuffled_projected
    )

    # Center representation
    center_features = model.center_embedding(
        center_tensor
    )

    # Class probabilities
    original_combined = torch.cat(
        [
            center_features,
            original_representation
        ],
        dim=0
    )

    shuffled_combined = torch.cat(
        [
            center_features,
            shuffled_representation
        ],
        dim=0
    )

    original_output = model.classifier(
        original_combined
    )

    shuffled_output = model.classifier(
        shuffled_combined
    )

    original_probability = torch.softmax(
        original_output,
        dim=0
    )

    shuffled_probability = torch.softmax(
        shuffled_output,
        dim=0
    )

difference = torch.abs(
    original_probability - shuffled_probability
)

print("Test dataset index:", i)
print("Number of neighbors:", neighbor_tensor.shape[0])

print("\nOriginal probabilities:")
print(original_probability)

print("\nShuffled probabilities:")
print(shuffled_probability)

print("\nMaximum difference:")
print(difference.max().item())

Test dataset index: 9
Number of neighbors: 5

Original probabilities:
tensor([9.9878e-01, 1.0788e-06, 1.3593e-06, 1.7905e-06, 4.1086e-08, 1.2176e-03])

Shuffled probabilities:
tensor([9.9756e-01, 1.3301e-06, 1.5567e-06, 2.1540e-06, 7.6053e-08, 2.4340e-03])

Maximum difference:
0.0012174248695373535


In [51]:
# test on 10 random permutations

differences = []

model.eval()

with torch.no_grad():

    projected_neighbors = model.input_projection(
        neighbor_tensor
    )

    # Fixed Janossy permutations
    torch.manual_seed(42)

    permutations = [
        torch.randperm(projected_neighbors.size(0))
        for _ in range(5)
    ]

    def get_janossy_probability(
        neighbor_features
    ):

        outputs = []

        for permutation in permutations:

            shuffled_features = neighbor_features[
                permutation
            ]

            shuffled_features = shuffled_features.unsqueeze(0)

            _, hidden = model.order_sensitive(
                shuffled_features
            )

            pooled = hidden[0, 0]

            outputs.append(pooled)

        representation = torch.stack(
            outputs
        ).mean(dim=0)

        center_features = model.center_embedding(
            center_tensor
        )

        combined = torch.cat(
            [
                center_features,
                representation
            ],
            dim=0
        )

        output = model.classifier(
            combined
        )

        return torch.softmax(
            output,
            dim=0
        )

    # Reference probability
    reference_probability = get_janossy_probability(
        projected_neighbors
    )

    for _ in range(10):

        shuffle = torch.randperm(
            projected_neighbors.size(0)
        )

        shuffled_neighbors = projected_neighbors[
            shuffle
        ]

        shuffled_probability = get_janossy_probability(
            shuffled_neighbors
        )

        difference = torch.abs(
            reference_probability
            - shuffled_probability
        ).max().item()

        differences.append(difference)

print("Permutation differences:")
print(differences)

print("\nMaximum difference:", max(differences))
print("Mean difference:", np.mean(differences))

Permutation differences:
[0.0012174248695373535, 0.0012804269790649414, 0.00027817487716674805, 0.0024791955947875977, 0.0006334170466288924, 2.5564921088516712e-05, 0.0013315081596374512, 0.0033507943153381348, 0.0020309090614318848, 0.0012157559394836426]

Maximum difference: 0.0033507943153381348
Mean difference: 0.0013843171764165163
